# qradiomics: Lung1 (NSCLC-Radiomics) survival

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/choilab-jefferson/qradiomics/blob/main/examples/lung1_survival_colab.ipynb)

End-to-end radiomics survival analysis on the public TCIA **NSCLC-Radiomics (Lung1)**
cohort (Aerts *et al.*, Nature Communications 2014), driven entirely by the `qr` CLI:

```
qr tcia download -> qr select validate -> qr convert from-manifest -> qr preprocess
  -> qr extract --engine rtools -> qr results merge -> qr analyze survival
```

* Public data only (TCIA). No credentials needed.
* Features come from the native `rtools` engine, a pure-Python port that installs
  everywhere. `pyradiomics` is optional (see the note after the extraction step).
* Runs on a small subset by default so it finishes in a Colab session. Set
  `N_PATIENTS = None` below for the full cohort.

## 0. Install

In [ ]:
# qradiomics from PyPI. The `rtstruct` extra adds rt-utils (needed by `qr convert rtstruct`).
!pip install -qU "qradiomics[rtstruct]"
!qr --version

## 1. Choose the cohort size and select paired series

`N_PATIENTS` is the knob: a small number keeps the demo quick, `None` runs all 422
patients (several hours and tens of GB of downloads). We keep only patients that have
both a CT series and an RTSTRUCT.

In [ ]:
import os
import pandas as pd

N_PATIENTS = 20  # subset size; None = the full cohort (422 patients)
WORK = "/content/lung1"
os.makedirs(WORK, exist_ok=True)

# List every CT / RTSTRUCT series of the collection.
!qr tcia series --collection NSCLC-Radiomics --modality CT --modality RTSTRUCT -o {WORK}/series.csv
series = pd.read_csv(f"{WORK}/series.csv")

paired = sorted(set(series.query("Modality == 'CT'").PatientID)
                & set(series.query("Modality == 'RTSTRUCT'").PatientID))
selected = paired if N_PATIENTS is None else paired[:N_PATIENTS]
sel = series[series.PatientID.isin(selected)]
sel.to_csv(f"{WORK}/selected_series.csv", index=False)
print(f"{len(selected)} patients, {len(sel)} series selected")

## 2. Download the DICOM data from TCIA

In [ ]:
!qr tcia download --manifest {WORK}/selected_series.csv -o {WORK}/dicom -j 8 --no-progress

## 3. Write and validate the pair manifest

A *pair manifest* is the contract between selection and the pipeline: one row per
(image, label) pair with the CT series directory, the RTSTRUCT file, its checksum and
the ROI name to look for (`GTV-1` is the primary tumour in Lung1). `qr select validate`
checks it and lists every problem before any compute is spent.

In [ ]:
import hashlib
from pathlib import Path

rows = []
for pid in selected:
    ct = sel[(sel.PatientID == pid) & (sel.Modality == "CT")].iloc[0]
    rs = sel[(sel.PatientID == pid) & (sel.Modality == "RTSTRUCT")].iloc[0]
    ct_dir = Path(WORK, "dicom", pid, ct.StudyInstanceUID, ct.SeriesInstanceUID)
    rs_files = sorted(Path(WORK, "dicom", pid, rs.StudyInstanceUID, rs.SeriesInstanceUID).glob("*.dcm"))
    ok = ct_dir.is_dir() and bool(rs_files)
    rows.append({
        "cohort": "LUNG1", "case_id": pid, "pair_id": f"{pid}_CT_manual",
        "image_key": "CT", "label_key": "manual",
        "series_uid": ct.SeriesInstanceUID, "series_dir": str(ct_dir),
        "rtstruct_path": str(rs_files[0]) if rs_files else "",
        "rtstruct_sha": hashlib.sha256(rs_files[0].read_bytes()).hexdigest() if rs_files else "",
        "roi_candidates": "GTV-1", "resample_mm": 1.0,
        "status": "ok" if ok else "skipped",
        "reason": "" if ok else "series not downloaded",
    })
pairs = pd.DataFrame(rows)
pairs.to_csv(f"{WORK}/pairs.csv", index=False)
print(pairs.status.value_counts().to_dict())

!qr select validate {WORK}/pairs.csv --check-paths

## 4. Convert, crop and resample

`qr convert from-manifest` turns each CT series into an NRRD volume and rasterises the
`GTV-1` contour into a label volume. `qr preprocess` then crops to the tumour (5 mm
margin) and resamples to 1 mm isotropic, which keeps memory low on a free Colab runtime.

A few Lung1 series cannot be read by every DICOM backend; such a pair is reported as
failed and simply left out of the later steps.

In [ ]:
!qr convert from-manifest -m {WORK}/pairs.csv -d {WORK}/nrrd -o {WORK}/nrrd_manifest.csv -j 4
!qr preprocess -m {WORK}/nrrd_manifest.csv -o {WORK}/pre --pad-mm 5 --jobs 2 --out-manifest {WORK}/pre_manifest.csv

## 5. Extract radiomics features (native `rtools` engine)

`--engine rtools` selects the pure-Python radiomics-tools port: 2D and 3D
intensity statistics, shape, and GLCM / GLRLM texture (97 features per patient),
with no compiled dependencies. It does not compute filtered (wavelet / LoG) images.

> **Optional: PyRadiomics.** `rtools` is the default engine (the flag above is kept
> explicit for clarity). PyRadiomics has no PyPI wheels for Python 3.10+, so
> it is not installed here. To use it, run
> `!pip install "pyradiomics @ git+https://github.com/AIM-Harvard/pyradiomics.git"`
> and pass `--engine pyradiomics` (or `--engine pyradiomics,rtools` for both). With
> `-p nsclc-survival --engine pyradiomics` it reproduces the larger Aerts feature set
> (Original + LoG + Wavelet, about 1,100 features per patient).

In [ ]:
!qr extract -m {WORK}/pre_manifest.csv --engine rtools -j 4 -o {WORK}/features.csv

features = pd.read_csv(f"{WORK}/features.csv")
print(f"{len(features)} patients x {features.shape[1] - 1} columns")
features.iloc[:5, :8]

## 6. Merge with the clinical table

The published Lung1 clinical table is joined on `PatientID`. In survival mode
`qr results merge` converts the survival time to months and writes two standard
columns, `OS_months` and `OS_event`.

In [ ]:
!curl -sSL -o {WORK}/clinical.csv https://www.cancerimagingarchive.net/wp-content/uploads/NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv

!qr results merge -f {WORK}/features.csv -c {WORK}/clinical.csv \
    --clinical-id-col PatientID --time-col Survival.time --event-col deadstatus.event \
    -o {WORK}/analysis_ready.csv

merged = pd.read_csv(f"{WORK}/analysis_ready.csv")
print(f"{len(merged)} patients merged, {int(merged.OS_event.sum())} events")
merged[["patient_id", "OS_months", "OS_event"]].head()

## 7. Univariate Cox proportional-hazards analysis

`qr analyze survival` fits a Cox model for every feature against overall survival.
With a small subset the ranking is noisy and few features reach significance; use a
larger `N_PATIENTS` for a more faithful reproduction of the published result.

In [ ]:
!qr analyze survival -i {WORK}/analysis_ready.csv --outcome OS_months --event OS_event \
    -o {WORK}/cox_results.csv --top-n 20

cox = pd.read_csv(f"{WORK}/cox_results.csv")
print(f"{len(cox)} features fitted; {(cox.p < 0.05).sum()} with p < 0.05 (uncorrected)")
cox.head(15)

## Notes

* This notebook uses the 97 unfiltered `rtools` features, so it demonstrates the
  workflow rather than reproducing the published signature, which used wavelet-filtered
  texture features (Aerts *et al.*, 2014). For that, install PyRadiomics (step 5) and
  run `qr extract -p nsclc-survival --engine pyradiomics`. The two engines use different feature names, so
  values and rankings differ between them.
* Citation: Aerts HJWL *et al.*, Nat Commun 2014;5:4006. Data: TCIA NSCLC-Radiomics
  collection (Clark *et al.*, J Digit Imaging 2013).

## Optional: drive this with an agent

Everything above runs by hand. If you would rather have an agent (Antigravity `agy`,
Claude Code, ...) run the same `qr` commands for you inside this Colab runtime,
install the small `nexus-lite` helper and prepare the environment. This section is
optional; skip it if you only want the notebook.

Agent skills and the full guide live in
[choilab-jefferson/nexus.lite](https://github.com/choilab-jefferson/nexus.lite)
(see `skills/nexus-colab`). The qradiomics skill that teaches an agent the `qr` CLI
ships with this repository under `.claude/skills/qradiomics/` and
`.agents/skills/qradiomics/`.

In [ ]:
!pip install -qU nexus-lite
from nexus_lite import colab
colab.ensure_env()